<br>

# Abordagem do LXML

_Script_ para obter dados usando LXML.

Quero simplificar usando o `POST`.

<br>

Michel Metran\
Data: 02.11.2025\
Atualizado em: 21.09.2026


In [ ]:
from pathlib import Path

from pyFDBS.logger import FBDSLogger
from pyFDBS.lxml import FBDS, download

<br>

---

## Pastas


In [ ]:
project_path = Path(".").absolute().parents[2]

# Diretório de saída
data_path = project_path / "data"

logs_path = data_path / "log"
logs_path.mkdir(parents=True, exist_ok=True)

output_path = data_path / "output"
output_path.mkdir(parents=True, exist_ok=True)
output_path

<br>

---

## FBDS

Instancia a classe


In [ ]:
# Instancia Logger
# Inicializa o logger uma única vez antes do loop
logger = FBDSLogger(
    log_dir=logs_path,
    new_session=True,
    console=False
)

# Instancia FBDS
fdbs = FBDS(
    temp_path=output_path,
    logger=logger,
)

<br>

---

## _Download_

Faz o _download_ dos _layers_ de um município.


In [ ]:
logger.start_download_session()

# Parâmetros
estado = "SP"

municipios = fdbs.get_municipalities(uf=estado)
municipios = municipios[0:1]

for muninicio in municipios:
    municipio_name = muninicio["name"]
    print(municipio_name)

    # Listas Layers Disponíveis
    lyrs = fdbs.get_layers(
        municipality=municipio_name,
        uf=estado,
    )
    for lyr in lyrs:
        layer_name = lyr["name"]
        print(layer_name)

        # Acessa o layer
        lyr = fdbs.get_layer(
            municipality=municipio_name,
            uf=estado,
            layer=layer_name,
        )

        # Lista de arquivos para download (do exemplo)
        files_to_download = fdbs.get_links(
            url=lyr["url"],
            ignore_first=2,
        )
        # print(files_to_download)

        # Download usando threads (melhor para downloads)
        results_thread = download.download_files_parallel(
            url_list=files_to_download,
            output_path=output_path,
            max_concurrent=4,
        )

# Finaliza a sessão após todo o processamento
logger.end_download_session()

<br>

---

## Lista _Links_

Ideal para ver os tipos de _shapefiles_ existentes


In [ ]:
estados = fdbs.states[0:10]
estados = fdbs.states[10:15]
# PE deu erro
estados = fdbs.states[16:25]
# estados = fdbs.states[15:20]
# estados = fdbs.states[20:25]
estados = fdbs.states[25:30]
estados

In [ ]:
# Parâmetros
# estado = "SP"
# municipios = municipios[0:2]

list_links = []

for estado in estados:
    # Lista Municipios
    municipios = fdbs.get_municipalities(uf=estado)
    # municipios = municipios[0:2]

    for muninicio in municipios:
        municipio_name = muninicio["name"]

        # Listas Layers Disponíveis
        lyrs = fdbs.get_layers(
            municipality=municipio_name,
            uf=estado,
        )
        for lyr in lyrs:
            layer_name = lyr["name"]
            print(estado, municipio_name, layer_name)

            # Acessa o layer
            lyr = fdbs.get_layer(
                municipality=municipio_name,
                uf=estado,
                layer=layer_name,
            )

            # Lista de arquivos
            if lyr["type"] != "file":
                files_to_download = fdbs.get_links(
                    url=lyr["url"],
                    ignore_first=2,
                )
                list_links.extend(files_to_download)

            else:
                print(lyr["type"])

In [ ]:
lyr

In [ ]:
list_links

In [ ]:
list_shps = [x["name"] for x in list_links if x["name"].endswith(".shp")]
list_shps

In [ ]:
list_shps = [x.split("_", maxsplit=2)[2] for x in list_shps]
list_shps

In [ ]:
list_shps = [x.replace(".shp", "") for x in list_shps]
list_shps = list(set(list_shps))
list_shps.sort()
list_shps

In [ ]:
a = [
    "APP",
    "APP_USO",
    "MASSAS_DAGUA",
    "NASCENTES",
    "RIOS_DUPLOS",
    "RIOS_SIMPLES",
    "USO",
]

b = [
    "APP",
    "APP_USO",
    "MASSAS_DAGUA",
    "MASSA_DAGUA",
    "NASCENTES",
    "RIOS_DUPLOS",
    "RIOS_DUPLOS_POL",
    "RIOS_SIMPLES",
    "USO",
]

c = [
    "APP",
    "APP_USO",
    "MASSAS_DAGUA",
    "NASCENTES",
    "RIOS_DUPLOS",
    "RIOS_SIMPLES",
    "USO",
]

d = [
    "APP",
    "APP_USO",
    "MASSAS_DAGUA",
    "MASSA_DAGUA",
    "MASSA_DAGUAS",
    "NASCENTE",
    "NASCENTES",
    "RIOS_DUPLOS",
    "RIOS_DUPLOS_",
    "RIOS_DUPLOS_POL",
    "RIOS_SIMPLES",
    "USO",
]


l = []
l.extend(a)
l.extend(b)
l.extend(c)
l.extend(d)
l = list(set(l))
l.sort()
l